# Four coupled transmons on IBM Quantum: VQE and VQD with Adam-SPSA

Load the best classical parameters, their **exact logical circuits and Pauli Hamiltonian**, then optimize ground and first-excited states using measured IBM Quantum objectives. Calculate the lowest collective transition from their physical energies. This is a self-contained hardware notebook: there are no presentation graphs, parameter sweeps, exact-diagonalization benchmarks or simulation-gradient tests.

| Original step | Implementation here |
|---|---|
| 3.1 Map | Load `four_transmon_best_parameters.json`, retaining its local-level basis, operator and parameter ordering. |
| 3.2 Optimize for target hardware | Select an IBM backend, compile the circuits and apply their layouts to the Hamiltonian. |
| 3.3 Execute | Run ground VQE and first-excited VQD with Adam-SPSA and independent candidate remeasurement. |
| 3.4 Report | Measure final physical energies and overlap, print $f_{01}$, and save parameters and job IDs. |

**Inputs and assumptions**

- First run the export cell in the classical four-transmon notebook and put its JSON in this notebook's working directory. Do not substitute the single-transmon JSON.
- The source model retains four eigenlevels per transmon: 256 states on **eight qubits**, both parity sectors, no unused padding states. The original defaults are 10 nH and 5.21015 GHz linear frequency per site, giving approximately 93.31 fF. Ring charge couplings remain illustrative.
- The exported Pauli operator is authoritative. It preserves the local eigenbasis convention used by the circuits; this notebook does not diagonalize or rebuild that basis.
- Units: energies are $E/h$ in GHz; circuit angles are radians. Adam-SPSA runs classically; the QPU supplies energy and overlap measurements. Hardware objectives use no statevectors or analytic simulator gradients.
- Compute-uncompute probability equals squared state overlap in the ideal limit. Gate/readout noise can bias it; the reported statistical intervals do not include all hardware bias. No shot-noise or hardware-accuracy guarantee is inferred from classical agreement.

**Execution:** the second code cell defaults to `mode="plan"`. Run all cells to inspect the inputs and compilation without submitting jobs or creating results files. To execute, configure an IBM saved account, change it to `mode="hardware"`, optionally choose a backend, and run all cells again. The hardware mode submits real jobs. It uses job mode rather than requiring session entitlement.

In [1]:
# %pip install qiskit==2.2.3 qiskit-ibm-runtime==0.43.1 numpy==2.3.4
# Locally checked with Python 3.14.0 and the versions above.
from __future__ import annotations
import base64
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
import importlib.metadata
import io
import json
import math
from pathlib import Path
from time import perf_counter
from types import SimpleNamespace
import numpy as np
import qiskit
from qiskit import QuantumCircuit, qpy
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import (
    QiskitRuntimeService,
    EstimatorV2 as RuntimeEstimator,
    SamplerV2 as RuntimeSampler,
)

## Initial settings

Physical parameters and the corrected effective capacitances come from the JSON. The settings below control this hardware experiment. The shot counts are configurable starting points, not sufficient-accuracy guarantees. Energy shots apply per measurement configuration; a Pauli Hamiltonian can need multiple configurations.

Use a small measurement-only first experiment by setting both step counts to zero. This still submits measurement jobs in hardware mode. Positive counts enable Adam-SPSA updates. There is no automatic retry or escalation of the measurement budget.

In [2]:
config = SimpleNamespace(
    mode="plan",  # "plan": no jobs; "hardware": submit IBM Quantum jobs.
    parameters=Path("four_transmon_best_parameters.json"),
    output=Path("four_transmon_ibm_results.json"),
    backend=None,  # Optional IBM backend name; None selects least-busy in hardware mode.
    account=None,  # Optional saved-account name; no API key belongs in this notebook.
    initial_layout=None,  # Optional list of eight physical qubit indices.
    seed=42,
    optimization_level=1,
    ground_steps=5,
    excited_steps=5,
    shots=2048,             # Training shots per energy configuration / overlap PUB.
    validation_shots=4096,  # Independent selection and final-readout shots.
    final_repeats=1,
    shortlist=2,
    selection_sigma=2.0,   # Keep warm start unless a remeasured improvement is resolved.
    learning_rate=0.01,    # Adam update coefficient [rad].
    perturbation=0.05,     # SPSA perturbation coefficient [rad].
    max_update_norm=0.05,  # Maximum angle-update norm [rad].
    resamplings=1,         # Independent SPSA directions per step.
    checkpoint_every=5,
    resilience_level=1,    # Estimator mitigation setting; sampler overlap remains raw.
    max_job_seconds=300,   # Maximum QPU seconds per job, not a total experiment limit.
)
if config.mode not in ("plan", "hardware"):
    raise ValueError('mode must be "plan" or "hardware".')
for name in ("shots", "validation_shots", "final_repeats", "shortlist", "resamplings",
             "checkpoint_every", "max_job_seconds"):
    value = getattr(config, name)
    if not isinstance(value, int) or value < 1:
        raise ValueError(f"{name} must be a positive integer.")
for name in ("ground_steps", "excited_steps", "seed"):
    value = getattr(config, name)
    if not isinstance(value, int) or value < 0:
        raise ValueError(f"{name} must be a nonnegative integer.")
for name in ("learning_rate", "perturbation", "max_update_norm"):
    value = getattr(config, name)
    if not np.isfinite(value) or value <= 0:
        raise ValueError(f"{name} must be finite and positive.")
if not np.isfinite(config.selection_sigma) or config.selection_sigma < 0:
    raise ValueError("selection_sigma must be finite and nonnegative.")
if config.optimization_level not in (0, 1, 2, 3) or config.resilience_level not in (0, 1, 2):
    raise ValueError("Unsupported transpiler or mitigation level.")
if config.parameters.resolve() == config.output.resolve():
    raise ValueError("Results must not overwrite the input JSON.")

### 3.1 Step 1: Map the problem to quantum circuits and operators

Read both state-preparation circuits and their named angle vectors. Load the Pauli sum in exactly the basis exported by the classical notebook. The Hamiltonian describes

$$H/h=\sum_i[4(E_{Ci}/h)\hat n_i^2-(E_{Ji}/h)\cos\hat\phi_i]
+\sum_{(i,j)}g_{ij}\hat n_i\hat n_j,$$

projected to four local levels per transmon. Its physical charge projection has already been performed in the classical notebook. Expect eight qubits, normally 255 angles per state and 77 Pauli terms with the original defaults. Input validation below checks file/circuit consistency; it does not perform a classical reference solve.

In [3]:
def write_json(path: Path, data: dict) -> None:
    """Checkpoint one results file, without writing figures or credentials."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(data, indent=2, allow_nan=False), encoding="utf-8")
    temporary.replace(path)

def versions() -> dict:
    result = {}
    for name in ("qiskit", "qiskit-ibm-runtime", "numpy", "scipy"):
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = "not installed"
    return result

@dataclass
class WarmState:
    circuit: QuantumCircuit
    angles: np.ndarray
    names: list[str]

    def values_for(self, circuit: QuantumCircuit, points: np.ndarray) -> np.ndarray:
        """Reorder logical angles to the actual transpiled parameter order."""
        if {p.name for p in circuit.parameters} != set(self.names):
            raise ValueError("Transpilation changed the circuit parameter set.")
        indices = [self.names.index(p.name) for p in circuit.parameters]
        return np.asarray(points, dtype=float)[:, indices]

    def bound(self, angles: np.ndarray) -> QuantumCircuit:
        by_name = dict(zip(self.names, angles))
        return self.circuit.assign_parameters({p: by_name[p.name] for p in self.circuit.parameters})

    def export(self, angles: np.ndarray) -> dict:
        buffer = io.BytesIO()
        qpy.dump(self.circuit, buffer)
        return {
            "parameter_order": self.names,
            "parameters_rad": np.asarray(angles, dtype=float).tolist(),
            "logical_circuit_qpy_base64": base64.b64encode(buffer.getvalue()).decode("ascii"),
        }


def load_four_transmon_export(path):
    data = json.loads(path.read_text(encoding="utf-8"))
    if data.get("schema_version") != 1 or data.get("kind") not in (
        "four_transmon_warm_start", "four_transmon_hardware_results"
    ):
        raise ValueError("Use the four-transmon JSON produced by the supplied export cell.")
    if data.get("kind") == "four_transmon_hardware_results" and data.get("status") != "completed":
        raise ValueError("A previous hardware results file must have completed before reuse.")
    model = data["model"]
    if model["num_transmons"] != 4 or model["local_levels"] != 4 or model["num_qubits"] != 8:
        raise ValueError("Expected four transmons, four levels per site, and eight qubits.")
    terms = data["hamiltonian"]["pauli_terms"]
    labels = [term["label"] for term in terms]
    coefficients = np.array([term["coefficient_GHz"] for term in terms], dtype=float)
    if not labels or not np.all(np.isfinite(coefficients)):
        raise ValueError("Pauli coefficients must be finite real values.")
    observable = SparsePauliOp.from_list(list(zip(labels, coefficients)))
    if observable.num_qubits != 8 or data["hamiltonian"]["num_qubits"] != 8:
        raise ValueError("Pauli operator and model widths disagree.")
    states = []
    for label in ("ground", "excited"):
        entry = data[label]
        circuits = qpy.load(io.BytesIO(base64.b64decode(entry["logical_circuit_qpy_base64"], validate=True)))
        if len(circuits) != 1:
            raise ValueError(f"Expected exactly one {label} circuit.")
        circuit, names = circuits[0], entry["parameter_order"]
        angles = np.asarray(entry["parameters_rad"], dtype=float)
        if angles.shape != (len(names),) or not np.all(np.isfinite(angles)):
            raise ValueError(f"Invalid {label} angle vector.")
        if len(names) != len(set(names)) or set(names) != {p.name for p in circuit.parameters}:
            raise ValueError(f"{label} parameter names do not match its circuit.")
        if circuit.num_qubits != 8 or circuit.num_clbits or any(
            item.operation.name in ("measure", "reset") for item in circuit.data
        ):
            raise ValueError("Expected eight-qubit unitary state-preparation circuits.")
        states.append(WarmState(circuit, angles, names))
    beta = float(data["beta_GHz"])
    if not np.isfinite(beta) or beta <= 0:
        raise ValueError("The VQD penalty must be finite and positive [GHz].")
    return data, states[0], states[1], observable, beta


data, ground, excited, hamiltonian, beta = load_four_transmon_export(config.parameters)
print(f"Loaded {hamiltonian.num_qubits} qubits and {len(hamiltonian)} Pauli terms.")
print(f"Ground/excited angles: {len(ground.angles)} / {len(excited.angles)} [rad]")
print("Effective capacitances [fF]:", data["model"]["C_effective_fF"])
print(f"VQD beta: {beta:.6f} GHz")

Loaded 8 qubits and 77 Pauli terms.
Ground/excited angles: 255 / 255 [rad]
Effective capacitances [fF]: [93.31249669455167, 93.31249669455167, 93.31249669455167, 93.31249669455167]
VQD beta: 6.754109 GHz


### 3.2 Step 2: Optimize for target hardware

Hardware mode loads your saved IBM account, selects an operational real backend with at least eight qubits, and transpiles for its actual instruction set and connectivity. A named backend and initial physical layout can be specified above. Each Hamiltonian receives the corresponding circuit layout.

Plan mode uses a generic line-connected target unless a named IBM backend is requested. It submits no primitive jobs. Compilation counts are useful resource estimates, not an accuracy prediction. Expect deeper circuits than the earlier ideal all-to-all target.

In [4]:
class Compiler:
    def __init__(self, backend, seed: int, level: int, layout: list[int] | None):
        self.backend, self.seed, self.level, self.layout = backend, seed, level, layout

    def run(self, circuit: QuantumCircuit) -> QuantumCircuit:
        pm = generate_preset_pass_manager(
            backend=self.backend, optimization_level=self.level,
            seed_transpiler=self.seed, initial_layout=self.layout,
        )
        compiled = pm.run(circuit)
        if self.layout is None:
            # Reuse the ground circuit's initial physical mapping for the other circuits.
            self.layout = compiled.layout.initial_index_layout(filter_ancillas=True)
        return compiled

def circuit_summary(circuit: QuantumCircuit) -> dict:
    counts = circuit.count_ops()
    two_qubit = sum(len(item.qubits) == 2 and item.operation.name != "barrier" for item in circuit.data)
    return {
        "depth": circuit.depth(), "two_qubit_gates": two_qubit,
        "operations": {name: int(number) for name, number in counts.items()},
        "initial_physical_qubits": circuit.layout.initial_index_layout(filter_ancillas=True),
    }

def overlap_circuit(excited: WarmState, ground: WarmState, ground_angles: np.ndarray) -> QuantumCircuit:
    """P(0...0) = |<psi_g|psi_e>|^2 for ideal compute-uncompute."""
    circuit = excited.circuit.copy()
    # Keep the preparation/uncompute boundary explicit during compilation.
    circuit.barrier()
    circuit.compose(ground.bound(ground_angles).inverse(), inplace=True)
    circuit.measure_all()
    return circuit

def get_backend(args, qubits: int):
    if args.mode == "hardware" or args.backend:
        service = QiskitRuntimeService(name=args.account) if args.account else QiskitRuntimeService()
        backend = service.backend(args.backend) if args.backend else service.least_busy(
            min_num_qubits=qubits, operational=True, simulator=False,
        )
        if backend.num_qubits < qubits or backend.configuration().simulator or not backend.status().operational:
            raise ValueError("Select an operational real processor with enough qubits.")
        return backend
    return GenericBackendV2(
        num_qubits=qubits, basis_gates=["rz", "sx", "x", "cx"],
        coupling_map=CouplingMap.from_line(qubits), seed=args.seed, noise_info=False,
    )


backend = get_backend(config, 8)
layout = config.initial_layout
if layout is not None and (
    len(layout) != 8 or len(set(layout)) != 8 or min(layout) < 0 or max(layout) >= backend.num_qubits
):
    raise ValueError("initial_layout must contain eight distinct valid physical qubit indices.")
compiler = Compiler(backend, config.seed, config.optimization_level, layout)
compile_started = perf_counter()
ground_isa = compiler.run(ground.circuit)
excited_isa = compiler.run(excited.circuit)
ground_hamiltonian_isa = hamiltonian.apply_layout(ground_isa.layout)
excited_hamiltonian_isa = hamiltonian.apply_layout(excited_isa.layout)
warm_overlap_isa = compiler.run(overlap_circuit(excited, ground, ground.angles))

report = {
    "schema_version": 1, "kind": "four_transmon_hardware_results",
    "qiskit_version": qiskit.__version__, "status": "planned", "mode": config.mode,
    "created_utc": datetime.now(timezone.utc).isoformat(), "versions": versions(),
    "backend": backend.name, "model": data["model"], "hamiltonian": data["hamiltonian"],
    "beta_GHz": beta,
    "ground": ground.export(ground.angles), "excited": excited.export(excited.angles),
    "settings": {k: str(v) if isinstance(v, Path) else v for k, v in vars(config).items()},
    "compiled": {"ground": circuit_summary(ground_isa), "excited": circuit_summary(excited_isa),
                 "warm_overlap": circuit_summary(warm_overlap_isa)},
    "compile_seconds": perf_counter() - compile_started,
    "jobs": [], "ground_history": [], "excited_history": [],
    "measurement_scope": "E/h in GHz; raw compute-uncompute counts. Statistical/repeat uncertainty excludes systematic hardware bias. Wall times include queue and client overhead.",
}
print(f"Selected target: {backend.name}; mode: {config.mode}")
for name, item in report["compiled"].items():
    print(f"{name}: depth {item['depth']}, two-qubit gates {item['two_qubit_gates']}")
if config.mode == "plan":
    print("Plan only: no jobs submitted and no results file created.")

Selected target: generic_backend_8q; mode: plan
ground: depth 1921, two-qubit gates 1653
excited: depth 2002, two-qubit gates 1752
warm_overlap: depth 4467, two-qubit gates 3754
Plan only: no jobs submitted and no results file created.


### 3.3 Step 3: Execute on target hardware

#### Measured energy and overlap costs

Ground VQE minimizes $E_0(\theta)=\langle H/h\rangle$. First-excited VQD minimizes

$$F_1(\theta)=\langle H/h\rangle+\beta\,p_0(\theta),\qquad
p_0(\theta)=|\langle\psi_g|\psi_e(\theta)\rangle|^2.$$

Runtime Estimator measures energy. Runtime Sampler measures the all-zero probability after excited preparation followed by the inverse of the selected ground circuit. No classical statevector projector is used. This equality for $p_0$ is ideal; hardware noise can bias the observed probability.

Every submitted job ID is checkpointed before waiting. There are no automatic retries. Estimator shot settings are per measurement configuration, not a total-shot budget across all Pauli terms and mitigation circuits. Wilson overlap intervals retain a nonzero uncertainty when zero successes are observed.

In [5]:
@dataclass
class Reading:
    energy_GHz: float
    energy_se_GHz: float
    overlap: float = 0.0
    overlap_se: float = 0.0
    overlap_successes: int = 0
    overlap_shots: int = 0

    def cost(self, beta: float) -> float:
        return self.energy_GHz + beta * self.overlap

    def cost_se(self, beta: float) -> float:
        return math.hypot(self.energy_se_GHz, beta * self.overlap_se)

def binomial_summary(successes: int, shots: int) -> tuple[float, float, list[float]]:
    """Return p, a nonzero boundary uncertainty, and a 95% Wilson interval."""
    p = successes / shots
    z = 1.959963984540054
    denominator = 1 + z**2 / shots
    center = (p + z**2 / (2 * shots)) / denominator
    half = z * math.sqrt(p * (1 - p) / shots + z**2 / (4 * shots**2)) / denominator
    # Wilson half-width/z remains nonzero even if no all-zero events are observed.
    return p, half / z, [max(0.0, center - half), min(1.0, center + half)]


class Measurements:
    """Hardware-only primitive adapter with parameter batching and job checkpoints."""
    def __init__(self, estimator, sampler, report, output):
        self.estimator, self.sampler = estimator, sampler
        self.report, self.output = report, output
        self.energy_points = 0
        self.overlap_points = 0

    def fail(self, exc):
        self.report["status"] = "interrupted" if isinstance(exc, KeyboardInterrupt) else "failed"
        self.report["error_type"] = type(exc).__name__
        write_json(self.output, self.report)

    def submit(self, primitive, pubs, kind, shots, label):
        try:
            if kind == "energy":
                primitive.options.default_shots = shots
                job = primitive.run(pubs)
            else:
                job = primitive.run(pubs, shots=shots)
            entry = {"kind": kind, "label": label, "job_id": str(job.job_id()),
                     "nominal_shots_per_configuration": shots, "status": "submitted"}
            self.report["jobs"].append(entry)
            write_json(self.output, self.report)
            return job, entry
        except BaseException as exc:
            self.fail(exc)
            raise

    def finish(self, submitted):
        job, entry = submitted
        try:
            result = job.result()
        except BaseException as exc:
            entry["status"] = "failed"
            self.fail(exc)
            raise
        entry["status"] = "completed"
        write_json(self.output, self.report)
        return result

    def energies(self, specs, shots, label):
        pubs = [(isa, op, state.values_for(isa, points)) for state, isa, op, points in specs]
        self.energy_points += sum(len(points) for _, _, _, points in specs)
        results = self.finish(self.submit(self.estimator, pubs, "energy", shots, label))
        values = []
        for pub, (_, _, _, points) in zip(results, specs):
            energies = np.asarray(pub.data.evs, dtype=float).ravel()
            errors = np.asarray(pub.data.stds, dtype=float).ravel()
            if len(energies) != len(points) or len(errors) != len(points):
                raise RuntimeError("Unexpected Estimator PUB shape.")
            if not np.all(np.isfinite(energies)) or not np.all(np.isfinite(errors)) or np.any(errors < 0):
                raise RuntimeError("Invalid Estimator energies or uncertainties.")
            values.append([Reading(float(e), float(s)) for e, s in zip(energies, errors)])
        return values

    def overlaps(self, state, isa, points, shots, label):
        self.overlap_points += len(points)
        pub = (isa, state.values_for(isa, points))
        result = self.finish(self.submit(self.sampler, [pub], "overlap", shots, label))[0]
        values = []
        for index in range(len(points)):
            counts = result.data.meas.get_counts(loc=index)
            values.append((counts.get("0" * state.circuit.num_qubits, 0), sum(counts.values())))
        return values


class Objective:
    def __init__(self, state: WarmState, isa: QuantumCircuit, observable: SparsePauliOp,
                 measurements: Measurements, beta: float = 0.0, overlap_isa=None):
        self.state, self.isa, self.observable = state, isa, observable
        self.measurements, self.beta, self.overlap_isa = measurements, beta, overlap_isa

    def evaluate(self, points, shots: int, label: str) -> list[Reading]:
        points = np.atleast_2d(np.asarray(points, dtype=float))
        energy = self.measurements.energies(
            [(self.state, self.isa, self.observable, points)], shots, label
        )[0]
        if self.overlap_isa is not None:
            overlaps = self.measurements.overlaps(self.state, self.overlap_isa, points, shots, label)
            for reading, (successes, total) in zip(energy, overlaps):
                p, se, _ = binomial_summary(successes, total)
                reading.overlap, reading.overlap_se = p, se
                reading.overlap_successes, reading.overlap_shots = successes, total
        return energy

#### Adam-SPSA optimizer and independent candidate selection

A random direction $\Delta_k\in\{-1,+1\}^{p}$ estimates all angle-gradient components from two measured costs:

$$\widehat g_k=\frac{F(\theta_k+c_k\Delta_k)-F(\theta_k-c_k\Delta_k)}{2c_k}\Delta_k.$$

Bias-corrected Adam moments update the angles. The implementation uses beta1 = 0.9, beta2 = 0.999, epsilon = $10^{-8}$, perturbation decay $k^{-0.101}$, learning-rate decay $k^{-0.1}$ and a configurable update-norm cap. These are starting settings rather than proven optimal hardware choices.

Record the warm start, perturbation probes and checkpoint iterates. Remeasure a shortlist independently, always including the warm start. Retain it unless a candidate's mean cost improves beyond `selection_sigma` times the combined uncertainty estimate. This guards against noisy minima but is not a formal confidence or global-optimum guarantee. Exact classical energies never choose candidates.

In [6]:
def adam_spsa(objective: Objective, start: np.ndarray, steps: int, args,
              stage: str, history: list) -> list[dict]:
    """Rademacher SPSA gradient estimates fed to bias-corrected Adam updates.

    Two objective values per random direction estimate all gradient components.
    This is not a parameter-shift gradient and uses no full-state information.
    Angles are not wrapped modulo 2*pi: controlled RY has a 4*pi period.
    """
    rng = np.random.default_rng(args.seed + (0 if stage == "ground" else 1000))
    theta = np.array(start, dtype=float, copy=True)
    m, v = np.zeros_like(theta), np.zeros_like(theta)
    candidates = []
    started = perf_counter()

    def keep(point, reading, iteration, kind):
        record = {
            "iteration": iteration, "kind": kind, "parameters_rad": point.tolist(),
            **asdict(reading), "cost_GHz": reading.cost(objective.beta),
            "cost_se_GHz": reading.cost_se(objective.beta),
        }
        candidates.append(record)
        return record

    baseline = objective.evaluate([theta], args.shots, f"{stage}:warm_start")[0]
    initial = keep(theta, baseline, 0, "warm_start")
    history.append(dict(initial, elapsed_seconds=perf_counter() - started))
    print(f"{stage}: warm-start cost = {initial['cost_GHz']:.9f} GHz", flush=True)
    for k in range(1, steps + 1):
        c_k = args.perturbation / k**0.101  # Perturbation scale [rad].
        directions = rng.choice([-1.0, 1.0], size=(args.resamplings, len(theta)))
        points = np.array([point for delta in directions for point in (theta + c_k*delta, theta - c_k*delta)])
        readings = objective.evaluate(points, args.shots, f"{stage}:gradient:{k}")
        costs = np.array([r.cost(objective.beta) for r in readings]).reshape(-1, 2)
        gradient = np.mean(((costs[:, 0] - costs[:, 1]) / (2*c_k))[:, None] * directions, axis=0)
        for point, reading in zip(points, readings):
            keep(point, reading, k, "spsa_probe")
        m = 0.9*m + 0.1*gradient
        v = 0.999*v + 0.001*gradient**2
        m_hat, v_hat = m/(1-0.9**k), v/(1-0.999**k)
        step = (args.learning_rate / k**0.1) * m_hat / (np.sqrt(v_hat) + 1e-8)
        # A modest update cap protects an already accurate classical warm start.
        norm = np.linalg.norm(step)
        if norm > args.max_update_norm:
            step *= args.max_update_norm/norm
        theta -= step
        if k % args.checkpoint_every == 0 or k == steps:
            reading = objective.evaluate([theta], args.shots, f"{stage}:checkpoint:{k}")[0]
            record = keep(theta, reading, k, "checkpoint")
            history.append(dict(record, elapsed_seconds=perf_counter() - started,
                                spsa_gradient_norm=float(np.linalg.norm(gradient))))
            print(f"{stage}: step {k}, cost = {record['cost_GHz']:.9f} GHz", flush=True)
    return candidates

def aggregate(readings: list[Reading], beta: float) -> dict:
    energies = np.array([r.energy_GHz for r in readings])
    within = math.sqrt(sum(r.energy_se_GHz**2 for r in readings))/len(readings)
    between = float(np.std(energies, ddof=1)/np.sqrt(len(readings))) if len(readings) > 1 else 0.0
    successes = sum(r.overlap_successes for r in readings)
    shots = sum(r.overlap_shots for r in readings)
    p, se, interval = binomial_summary(successes, shots) if shots else (0.0, 0.0, [0.0, 0.0])
    energy, energy_se = float(np.mean(energies)), max(within, between)
    return {
        "energy_GHz": energy, "energy_se_GHz": energy_se,
        "raw_compute_uncompute_probability": p if shots else None,
        "overlap_se": se if shots else None,
        "overlap_wilson_95_interval": interval if shots else None, "overlap_shots": shots,
        "cost_GHz": energy + beta*p, "cost_se_GHz": math.hypot(energy_se, beta*se),
    }

def select_remeasured(objective: Objective, candidates: list[dict], args, stage: str) -> dict:
    """Include the warm start and independently remeasure a noisy-cost shortlist."""
    shortlist = [candidates[0]]
    seen = {tuple(candidates[0]["parameters_rad"])}
    for candidate in sorted(candidates, key=lambda c: c["cost_GHz"]):
        key = tuple(candidate["parameters_rad"])
        if key not in seen and len(shortlist) < args.shortlist:
            shortlist.append(candidate)
            seen.add(key)
    points = np.array([c["parameters_rad"] for c in shortlist for _ in range(args.final_repeats)])
    readings = objective.evaluate(points, args.validation_shots, f"{stage}:independent_selection")
    assessed = []
    for index, candidate in enumerate(shortlist):
        subset = readings[index*args.final_repeats:(index+1)*args.final_repeats]
        assessed.append({"parameters_rad": candidate["parameters_rad"],
                         "training_origin": candidate["kind"], **aggregate(subset, objective.beta)})
    baseline = assessed[0]
    eligible = [r for r in assessed[1:] if r["cost_GHz"] < baseline["cost_GHz"]
                - args.selection_sigma*math.hypot(r["cost_se_GHz"], baseline["cost_se_GHz"])]
    selected = min(eligible, key=lambda r: r["cost_GHz"]) if eligible else baseline
    return {"selected": selected, "shortlist": assessed,
            "criterion": "Lowest independently remeasured mean cost with a statistical improvement guard relative to the warm start. This heuristic is not a global optimum or confidence guarantee.",
            "selection_sigma": args.selection_sigma}

#### Run ground-state VQE

Initialize from the exported ground angles. Runtime energy measurements drive Adam-SPSA, followed by independent candidate selection. Save the selected logical circuit and parameters for the ground reference used by VQD.

Plan mode skips primitive construction and all objective calls. Hardware mode creates or checkpoints the configured results JSON. Saved credentials are loaded by the backend-selection step; API keys are never embedded in the notebook or results.

In [7]:
if config.mode == "hardware":
    report["status"] = "running"
    write_json(config.output, report)
    execution_started = perf_counter()
    estimator = RuntimeEstimator(mode=backend, options={
        "default_shots": config.shots, "resilience_level": config.resilience_level,
        "max_execution_time": config.max_job_seconds,
    })
    sampler = RuntimeSampler(mode=backend, options={
        "default_shots": config.shots, "max_execution_time": config.max_job_seconds,
    })
    measurements = Measurements(estimator, sampler, report, config.output)
    try:
        ground_objective = Objective(ground, ground_isa, ground_hamiltonian_isa, measurements)
        candidates = adam_spsa(ground_objective, ground.angles, config.ground_steps,
                               config, "ground", report["ground_history"])
        report["ground_selection"] = select_remeasured(ground_objective, candidates, config, "ground")
        ground_angles = np.array(report["ground_selection"]["selected"]["parameters_rad"])
        report["ground"] = ground.export(ground_angles)
        report["status"] = "ground_selected"
        write_json(config.output, report)
    except BaseException as exc:
        measurements.fail(exc)
        raise
else:
    ground_angles = ground.angles.copy()
    print("Plan only: VQE skipped; exported ground angles retained for inspection.")

Plan only: VQE skipped; exported ground angles retained for inspection.


#### Run first-excited-state VQD

Freeze the **hardware-selected ground circuit parameters** and recompile the compute-uncompute circuit. Initialize the excited circuit from its exported angles, then optimize the measured deflated cost using Adam-SPSA. Select by deflated cost, not by the smallest raw energy, which could favor the ground state.

The original penalty is retained from the classical export. Its sufficiency for the real experiment depends on ground-state quality and noisy overlap measurements. A completed optimization is not by itself proof that the first excited state was found.

In [8]:
if config.mode == "hardware":
    try:
        overlap_isa = compiler.run(overlap_circuit(excited, ground, ground_angles))
        report["compiled"]["selected_ground_overlap"] = circuit_summary(overlap_isa)
        excited_objective = Objective(excited, excited_isa, excited_hamiltonian_isa,
                                      measurements, beta=beta, overlap_isa=overlap_isa)
        candidates = adam_spsa(excited_objective, excited.angles, config.excited_steps,
                               config, "excited", report["excited_history"])
        report["excited_selection"] = select_remeasured(excited_objective, candidates, config, "excited")
        excited_angles = np.array(report["excited_selection"]["selected"]["parameters_rad"])
        report["excited"] = excited.export(excited_angles)
        report["status"] = "states_selected"
        write_json(config.output, report)
    except BaseException as exc:
        measurements.fail(exc)
        raise
else:
    excited_angles = excited.angles.copy()
    print("Plan only: VQD skipped; no energy or overlap jobs submitted.")

Plan only: VQD skipped; no energy or overlap jobs submitted.


### 3.4 Step 4: Report final energies and save parameters

Fresh final measurements, separate from candidate selection, supply the physical energies and raw overlap probability. Since energies are already stored as $E/h$ in GHz,

$$f_{01}=E_{1,\mathrm{GHz}}-E_{0,\mathrm{GHz}}.$$

The artificial VQD penalty is excluded from the transition. Reported energy uncertainty combines primitive errors and repeat variability; the transition uncertainty assumes independent energy sampling. These estimates do not include all gate/readout bias, drift or effective-model uncertainty.

The results JSON retains both selected circuits and angle mappings, the same Pauli Hamiltonian, job IDs, histories and final readouts. It can be used as a future input after a completed run. No graph, charge-cutoff sweep, statevector reconstruction or classical benchmarking is performed.

In [9]:
if config.mode == "hardware":
    try:
        repeats = config.final_repeats
        readings = measurements.energies([
            (ground, ground_isa, ground_hamiltonian_isa, np.tile(ground_angles, (repeats, 1))),
            (excited, excited_isa, excited_hamiltonian_isa, np.tile(excited_angles, (repeats, 1))),
        ], config.validation_shots, "fresh_final_energies")
        ground_final, excited_final = (aggregate(values, 0.0) for values in readings)
        overlap_counts = measurements.overlaps(excited, overlap_isa,
            np.tile(excited_angles, (repeats, 1)), config.validation_shots, "fresh_final_overlap")
        successes = sum(pair[0] for pair in overlap_counts)
        shots = sum(pair[1] for pair in overlap_counts)
        overlap, overlap_se, overlap_interval = binomial_summary(successes, shots)
        excited_final.update({
            "raw_compute_uncompute_probability": overlap, "overlap_se": overlap_se,
            "overlap_wilson_95_interval": overlap_interval, "overlap_shots": shots,
            "cost_GHz": excited_final["energy_GHz"] + beta*overlap,
            "cost_se_GHz": math.hypot(excited_final["energy_se_GHz"], beta*overlap_se),
        })
        transition = excited_final["energy_GHz"] - ground_final["energy_GHz"]
        transition_se = math.hypot(ground_final["energy_se_GHz"], excited_final["energy_se_GHz"])
        report["ground"]["energy_GHz"] = ground_final["energy_GHz"]
        report["excited"]["energy_GHz"] = excited_final["energy_GHz"]
        report["transition_GHz"] = transition
        report["final"] = {
            "ground": ground_final, "excited": excited_final,
            "transition_GHz": transition, "transition_se_GHz": transition_se,
            "raw_compute_uncompute_probability": overlap,
            "overlap_wilson_95_interval": overlap_interval,
            "uncertainty_scope": "Statistical/repeat estimates; not full hardware or physical-model uncertainty.",
        }
        report["energy_parameter_points"] = measurements.energy_points
        report["overlap_parameter_points"] = measurements.overlap_points
        report["wall_seconds"] = perf_counter() - execution_started
        report["status"] = "completed"
        write_json(config.output, report)
        print(f"E0/h = {ground_final['energy_GHz']:.9f} +/- {ground_final['energy_se_GHz']:.3g} GHz")
        print(f"E1/h = {excited_final['energy_GHz']:.9f} +/- {excited_final['energy_se_GHz']:.3g} GHz")
        print(f"Lowest collective f01 = {transition:.9f} +/- {transition_se:.3g} GHz (statistical)")
        print(f"Raw overlap probability: {overlap:.6f}; 95% Wilson interval: {overlap_interval}")
        print(f"Saved parameters, measurements and job IDs: {config.output.resolve()}")
    except BaseException as exc:
        measurements.fail(exc)
        raise
else:
    print("Plan complete. No QPU measurements or results files were produced.")
    print('To execute, configure your IBM account, change config.mode to "hardware", and rerun all cells.')

Plan complete. No QPU measurements or results files were produced.
To execute, configure your IBM account, change config.mode to "hardware", and rerun all cells.


## Scope and sources

This notebook retains the original eight-qubit encoding and ansätze. It does not claim that these deep circuits achieve the classical notebook's accuracy on a noisy processor. Raw compute-uncompute probabilities can be distorted by gate and readout noise; a small observed probability is not a proof of orthogonality. Actual QPU execution must be assessed with your account and selected backend.

Locally checked with Qiskit 2.2.3 and Runtime 0.43.1. The delivered default is a no-submission plan. Offline verification of the file/Runtime interfaces is separate from this notebook and is not a QPU experiment.

- [IBM EstimatorV2 API for the tested version](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/0.43/estimator-v2): PUBs, backend execution and observable layouts.
- [IBM SamplerV2 API for the tested version](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/0.43/sampler-v2): measured bitstrings and shots.
- [IBM account setup](https://quantum.cloud.ibm.com/docs/en/ibm-quantum-compute/save-credentials): configure credentials separately; no token belongs in this notebook.
- [Adam-SPSA research](https://arxiv.org/abs/1910.03591) and [Adam](https://arxiv.org/abs/1412.6980): randomized perturbation gradients and bias-corrected adaptive moments. Schedules and the update cap above are explicitly chosen implementation settings.
- [VQD](https://arxiv.org/abs/1805.08138): deflation with an overlap penalty.